<a href="https://colab.research.google.com/github/DanielV-13/FNPE---CALCULO-DE-PARAMETROS-de-MODELO-DE-LA-GOTA-LIQUIDA-SEMF-/blob/main/FNPE_MODELO_DE_LA_GOTA_LIQUIDA_(SEMF).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica Computacional: Modelo de la Gota Líquida y Fórmula Semiempírica de Masas (SEMF)

## 📌 Contexto General y Objetivos
En esta práctica de **Física Nuclear y de Partículas**, analizamos las propiedades macroscópicas y microscópicas de los núcleos atómicos utilizando un enfoque semiclásico conocido como el **Modelo de la Gota Líquida** (propuesto por George Gamow y desarrollado por Carl Friedrich von Weizsäcker)[cite: 18].

El objetivo principal es comprobar cómo una serie de términos físicos e interacciones fundamentales (la fuerza nuclear fuerte, la repulsión culombiana, el Principio de Exclusión de Pauli y el apareamiento de espines) logran explicar con gran precisión la estabilidad y la energía de enlace de los núcleos atómicos[cite: 18, 21].

## 📂 Origen de los Datos
* **Fuente Oficial:** Portal del *National Nuclear Data Center* (NNDC), utilizando la base de datos de las **Nuclear Wallet Cards** (`https://www.nndc.bnl.gov/walletcards/search.html`)[cite: 2, 11].
* **Variables extraídas del archivo (`walletcards.csv`):**
  * `Atomic Number (Z)`: Número de protones (define el elemento químico).
  * `Atomic Mass (A)`: Número total de nucleones (protones + neutrones).
  * `Mass Excess`: Exceso de masa experimental necesario para calcular las energías de enlace reales.
  * `Half-Life`: Vida media del nucleido, utilizada para filtrar y clasificar los núcleos en **estables** (`"STABLE"`) e **inestables**.

In [ ]:
# Importación de librerías esenciales para análisis numérico y visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuración estética para las gráficas
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 12

# 1. Cargar el dataset oficial de las Nuclear Wallet Cards
df = pd.read_csv('walletcards.csv')

# 2. Filtrar únicamente los estados fundamentales (Level Index == 0) para evitar estados excitados
df_base = df[df['Level Index'] == 0].copy()

# 3. Identificar núcleos estables basados en la columna Half-Life
df_base['is_stable'] = df_base['Half-Life'].astype(str).str.upper() == 'STABLE'

# Mostrar un resumen rápido de los datos cargados
print(f"Total de registros analizados en el estado fundamental: {len(df_base)}")
print(f"Núcleos estables identificados: {df_base['is_stable'].sum()}")
df_base.head()

## ⚛️ Fundamento Físico: ¿Cómo calculamos la Energía de Enlace ($B$)?

Experimentalmente, un núcleo atómico pesa *menos* que la suma individual de sus protones y neutrones por separado. Esa "masa perdida" es la energía de enlace $B$ que mantiene unidas a las partículas mediante la **fuerza nuclear fuerte**:

$$B(Z, A) = [Z \cdot m_p + N \cdot m_n - M_{\text{núcleo}}] \cdot c^2$$

En las tablas de datos nucleares, este déficit o exceso de masa se cuantifica a través del **Exceso de Masa ($\Delta$)**. A partir de él, calculamos la energía de enlace total y la dividimos entre $A$ para obtener la **energía de enlace por nucleón ($B/A$)**, que es la métrica estándar de estabilidad en física nuclear.

In [ ]:
ASAS